In [48]:
%reload_ext autoreload
%autoreload 2

import sys
sys.path.append('/hildafs/home/nianyic/utils')
from load_basics import *
from astrid_utils import *
from bhdetails_util import *
hh = 0.6774
cosmo = load_astrid_cosmo()
import pickle


In [8]:
mergers = np.load("/hildafs/home/nianyic/scratch1/Astrid_data/lisa_mbhcat/lisa_mcat_v3.0_nogal.npy")

In [9]:
print(len(mergers))
print(mergers.dtype)

3685214
[('z', '<f8'), ('ID1', '<i8'), ('ID2', '<i8'), ('m1', '<f8'), ('m2', '<f8'), ('mdot1', '<f8'), ('mdot2', '<f8'), ('pos1', '<f8', (3,)), ('pos2', '<f8', (3,)), ('v1', '<f8', (3,)), ('v2', '<f8', (3,))]


In [39]:
part = BigFile("/hildafs/datasets/Asterix/PART_files/PART_817")
id5 = part["5/ID"][:]
tsw = part["5/BlackholeSwallowTime"][:]
swid = part["5/BlackholeSwallowID"][:]
sw = part["5/Swallowed"][:]
mask = (sw > 0).nonzero()[0]

In [40]:
print(len(mask) / len(id5))
print(len(mask))

id1 = swid[mask]
id2 = id5[mask]
zmerge = 1/tsw[mask] - 1

print(zmerge[:10])

0.23730430038466585
3779809
[0.31947315 2.7920735  2.4848855  2.194493   2.3850803  3.107417
 3.6146998  3.198399   4.546158   3.6277714 ]


In [27]:
dt = np.dtype([("id1", "q"), ("id2", "q"), ("zmerge", "d")])
mergers = np.zeros(len(mask), dtype=dt)
mergers["id1"] = id1
mergers["id2"] = id2
mergers["zmerge"] = zmerge
print(len(mergers))

3779809


In [57]:
snap = 810
bf = BigFile("/hildafs/datasets/Asterix/BHdetails-sorted/BH-Details-R%03d"%snap)

zz = bf["z"][:]
zmin, zmax = zz.min(), zz.max()
print(zmin, zmax)


0.0 0.004714163


In [42]:
mask = mergers["zmerge"] >= zmin
mask &= mergers["zmerge"] <= zmax
mthis = mergers[mask]

In [43]:
mm = mthis[1938]
id1, id2, zmerge = mm["id1"], mm["id2"], mm["zmerge"]
print(zmerge)

mask1 = (BHID == id1).nonzero()[0]
zz1 = zz[mask1]
mass1 = BHMass[mask1]
acc1 = acBHMass[mask1]

mask2 = (BHID == id2).nonzero()[0]
zz2 = zz[mask2]
mass2 = BHMass[mask2]

0.002354264259338379


In [44]:
print(sum(mask1), sum(mask2))

89 42


In [49]:
bhinfo = BHinfo()

Total number of snaps:  221
first snap:  5
last snap:  810
Total number of mergers:  3488673
Redshift range:  0.14800487458705902 10.265210307034993


In [51]:
with open(bhinfo.idx_dir + "/id2idx.pkl", 'rb') as f:
    id2idx = pickle.load(f)
    
Nsnaps = len(bhinfo.snaps)
SearchTableColNames = [str(snap) for snap in bhinfo.snaps]
SearchTableColTypes = [np.int64] * Nsnaps
SearchTableDataType = np.dtype(list(zip(SearchTableColNames, SearchTableColTypes)))
SearchTableElemSize = np.zeros(1, dtype=SearchTableDataType).nbytes
print("Size of each element in bytes:", SearchTableElemSize)
# print("Loading Length and Offset for %d BHs..."%len(bhidlist), flush=True)

Size of each element in bytes: 1768


In [54]:
idx1 = id2idx[id1]
idx2 = id2idx[id2]

length1 = np.fromfile(bhinfo.lenfile, dtype=SearchTableDataType, count=1, offset = idx1 * SearchTableElemSize)[0]
offset1 = np.fromfile(bhinfo.offfile, dtype=SearchTableDataType, count=1, offset = idx1 * SearchTableElemSize)[0]


length2 = np.fromfile(bhinfo.lenfile, dtype=SearchTableDataType, count=1, offset = idx2 * SearchTableElemSize)[0]
offset2 = np.fromfile(bhinfo.offfile, dtype=SearchTableDataType, count=1, offset = idx2 * SearchTableElemSize)[0]

In [56]:
print(length1)

(0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 50, 34, 43, 44, 72, 50, 45, 19, 33, 28, 42, 33, 20, 41, 41, 33, 30, 18, 17, 33, 25, 30, 26, 34, 36, 40, 22, 32, 38, 41, 25, 68, 49, 38, 54, 28, 58, 34, 28, 49, 77, 74, 52, 58, 38, 38, 49, 47, 49, 83, 55, 95, 71, 26, 59, 54, 68, 49, 51, 58, 76, 88, 61, 40, 35, 59, 29, 45, 47, 44, 61, 32, 56, 98, 85, 54, 115, 57, 57, 74, 114, 83, 25, 64, 56, 62, 89, 141, 62, 70, 85, 143, 22, 27, 80, 46, 191, 72, 88, 44, 126, 114, 88, 37, 106, 63, 169, 76, 104, 20, 169, 95, 93, 70, 46, 237, 153, 69, 70, 173, 129, 83, 65, 93, 35, 105, 123, 89, 137, 150, 91, 121, 157, 83, 191, 199, 79, 177, 200, 85, 129, 193, 95, 107, 132, 65, 47, 35, 46, 154, 173, 111, 59, 112, 148, 91, 143, 152, 96, 147, 106, 92, 82, 69, 89)
